# Tahap 0 — kelayakan sirkularisasi circRNA-HBsAg (PIE, IRES CVB3)

Notebook ini menjalankan keputusan **GO / NO-GO** langkah demi langkah dan
menjelaskan setiap angka yang keluar. Tujuannya dua: menjadi dokumentasi metode
yang dapat dibaca pembimbing, dan menjadi tempat Anda mencatat provenans
sekuens.

**Sebelum mulai, isi sel "Catatan provenans" di bawah.** Tanpa itu, laporan
tetap jalan, tetapi dokumen tugas akhir Anda kehilangan jejak sumber data.

Lisensi alat: ViennaRNA (ViennaRNA License, gratis akademik), primer3 (GPLv2),
Optuna (MIT), paket `circhbsag` ini (MIT). Jalankan sel pertama untuk melihat
versi yang benar-benar terpasang.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import circhbsag
from circhbsag import (access, bsj_stats, codon, construct, fold, motifs,
                       parts as parts_mod, primers, report, score)

print("circhbsag", circhbsag.__version__)
print(json.dumps(circhbsag.licenses(), indent=2, ensure_ascii=False))

## 0. Catatan provenans — ISI SENDIRI

Ubah sel di bawah menjadi catatan yang benar untuk sekuens yang Anda pakai.
Nilai ini hanya dicetak ke laporan; pipeline tidak memverifikasinya.

In [ ]:
CATATAN_PROVENANS = {
    "tanggal_unduh": "ISI_SENDIRI",            # mis. "2026-11-20"
    "ires_cvb3": {"aksesi": "ISI_SENDIRI", "koordinat": "ISI_SENDIRI"},
    "intron": {"sumber": "ISI_SENDIRI", "aksesi_atau_plasmid": "ISI_SENDIRI"},
    "hbsag": {"aksesi": "ISI_SENDIRI", "genotipe": "B", "panjang_aa": None},
    "tabel_kodon": "bawaan [PERKIRAAN] / diganti sendiri — pilih salah satu",
    "operator": "ISI_SENDIRI",
}
for k, v in CATATAN_PROVENANS.items():
    print(f"{k:16s} {v}")

## 1. Muat bagian konstruk

Dua konfigurasi tersedia:

| Konfigurasi | Isi | Label laporan |
|---|---|---|
| `config/parts.yaml` | sekuens Anda sendiri, `provenance_policy: strict` | `[HASIL IN SILICO]` |
| `config/parts.example.yaml` | sekuens acak sintetis | `[UJI PERANGKAT LUNAK]` |

Notebook ini mencoba konfigurasi sungguhan lebih dahulu dan **jatuh ke
konfigurasi contoh hanya bila berkas referensi belum ada** — sambil mencetak
peringatan besar, supaya Anda tidak pernah tidak sengaja melaporkan angka uji
sebagai hasil.

In [ ]:
REAL = ROOT / "config" / "parts.yaml"
EXAMPLE = ROOT / "config" / "parts.example.yaml"

try:
    parts, meta = parts_mod.load_parts(REAL)
    CFG = REAL
    print("Memakai konfigurasi SUNGGUHAN:", REAL.name)
except Exception as exc:
    print("Konfigurasi sungguhan tidak dapat dipakai:")
    print(" ", exc)
    print()
    print("=" * 78)
    print("JATUH KE DATA CONTOH SINTETIS. Angka di notebook ini TIDAK BOLEH")
    print("dikutip sebagai hasil ilmiah. Isi data/reference/ lalu jalankan ulang.")
    print("=" * 78)
    parts, meta = parts_mod.load_parts(EXAMPLE)
    CFG = EXAMPLE

print()
for name, p in meta["parts"].items():
    print(f"{name:16s} {p['length']:6d} nt  {p['provenance_status']:24s} {p.get('accession') or p.get('source') or ''}")
print()
print("Boleh dilaporkan sebagai hasil ilmiah:", meta["results_may_be_reported_as_scientific"])

## 2. Optimasi kodon ORF HBsAg

Objektif gabungan: CAI tinggi, GC mendekati target, GC jendela 30–70%, tanpa
situs BsaI/BsmBI, tanpa motif imunostimulan/destabilisasi, dan kerapatan m6A
RRACH rendah. Hill-climbing stokastik dengan **seed tetap**, jadi hasilnya
dapat diulang.

Pembanding yang dicetak adalah **baseline naif**: memilih kodon paling disukai
inang untuk setiap residu. Baseline itu selalu punya CAI = 1,00 tetapi biasanya
GC terlalu tinggi dan situs m6A lebih banyak — itulah trade-off yang dioptimasi.

In [ ]:
protein, pmeta = parts_mod.load_protein(CFG)
print(f"Protein {pmeta.get('length_aa', len(protein))} aa, sintetis={pmeta.get('synthetic')}")
print(protein[:60], "...")

hasil_kodon = codon.optimize(protein, iterations=20000, seed=20260101)

print("\nHasil optimasi:")
for k, v in hasil_kodon["cost"]["metrics"].items():
    print(f"  {k:24s} {v}")
print("\nBaseline naif (kodon paling disukai saja):")
for k, v in hasil_kodon["baseline_naive"].items():
    print(f"  {k:24s} {v}")

ORF = hasil_kodon["orf_dna"]
print(f"\nORF {len(ORF)} nt, mulai {ORF[:3]}, akhir {ORF[-3:]}")

## 3. Rakit prekursor dan lingkaran

Tata letak prekursor dan lingkaran yang dihasilkan:

```
prekursor:  T7 | GGG | HA5 | intron 3' | E2 | spacer5 | IRES | ORF | spacer3 | E1 | intron 5' | HA3
lingkaran:                               E2 | spacer5 | IRES | ORF | spacer3 | E1
```

Sambungan balik (BSJ) ada di antara basa terakhir E1 dan basa pertama E2.

In [ ]:
parts["ORF"] = construct.Part("ORF", ORF, role="ORF teroptimasi kodon")
asm = construct.build(parts)

print(f"Prekursor : {len(asm.precursor):5d} nt")
print(f"Lingkaran : {len(asm.circle):5d} nt")
print(f"BSJ       : posisi {asm.bsj_position} (basa terakhir E1)")
print("\nKoordinat pada lingkaran (1-based, inklusif):")
for name, (a, b) in asm.circle_coords.items():
    print(f"  {name:20s} {a:5d} .. {b:5d}   ({b - a + 1} nt)")

tpl = construct.permuted_dna_template(asm)
print("\nPemeriksaan templat IVT:")
print("  promotor T7 ditemukan :", tpl["t7_promoter_found"])
print("  tiga nt pertama       :", tpl["first_three_nt"], "(GGG = inisiasi T7 paling efisien)")
for n in tpl["notes"]:
    print("  CATATAN:", n)

## 4. Lipatan sirkular pada 37 dan 55 °C

Dua suhu dipakai karena keduanya menuntut sifat berbeda:

- **55 °C** — kondisi reaksi splicing PIE in vitro. Yang penting di sini: situs
  splice dan heliks P1/P10 **terbuka** agar ribozim intron grup I dapat bekerja.
- **37 °C** — kondisi di dalam sel. Yang penting di sini: IRES **terlipat
  benar**, start codon dapat dicapai, dan tidak ada dsRNA panjang.

Setiap kriteria nanti memakai **nilai terburuk** dari kedua suhu.

In [ ]:
for nama, r in fold.fold_profile(asm.circle, (37.0, 55.0), circular=True).items():
    print(f"Lingkaran @{nama}: MFE {r.mfe_kcal:8.2f} kcal/mol  "
          f"({r.mfe_per_nt:.4f}/nt)  ensemble {r.ensemble_free_energy_kcal:8.2f}  "
          f"keragaman {r.ensemble_diversity:7.2f}  freq(MFE) {r.frequency_mfe_in_ensemble:.4g}")

print()
for nama, r in fold.fold_profile(asm.precursor, (37.0, 55.0), circular=False).items():
    print(f"Prekursor @{nama}: MFE {r.mfe_kcal:8.2f} kcal/mol  ({r.mfe_per_nt:.4f}/nt)")

print("""
Cara membaca:
  MFE/nt      lebih negatif = lipatan lebih rapat. Terlalu longgar (> -0,25)
              berarti lingkaran tidak kompak; terlalu rapat menaikkan risiko
              dsRNA panjang dan situs yang tertutup.
  keragaman   mean base-pair distance dalam ensemble. Nilai besar relatif
              terhadap panjang = banyak lipatan alternatif, prediksi kurang kokoh.
  freq(MFE)   proporsi ensemble yang menempati struktur MFE.
""")

## 5. Probabilitas pasangan basa di sambungan balik

Jendela ±10 nt di sekitar BSJ, **melingkar melewati batas sekuens** (inilah
sebabnya lipatan harus dihitung dengan mode sirkular, bukan linear). Yang
dilaporkan adalah P(tak-berpasangan) rata-rata: makin tinggi, makin terbuka
sambungannya.

In [ ]:
for t in (37.0, 55.0):
    r = fold.fold(asm.circle, circular=True, temperature_c=t)
    j = fold.junction_pairing(r, asm.bsj_position, span=10)
    print(f"@{t:g} C  P(tak-berpasangan) rerata = {j['mean_unpaired_probability']:.4f}")
    print(f"        struktur jendela: {j['structure_window']}")
    print("        (titik = tak-berpasangan, kurung = berpasangan)")

## 6. Aksesibilitas situs splice dan P1/P10 dengan RNAplfold

RNAplfold menghitung peluang satu jendela beruntun tetap **tak-berpasangan**
dalam ansambel lipatan lokal. Wilayah yang diukur:

- `splice5_site`, `splice3_site` — batas eksón/intron, tempat transesterifikasi
- `intron_5half_IGS`, `intron_3half_IGS` — *internal guide sequence* yang
  membentuk heliks **P1** (dengan ujung 3' E1) dan **P10** (dengan awal E2)

**[LITERATUR]** Wesselhoeft dkk. (2018) menunjukkan spacer yang merusak struktur
kedua separuh intron menghilangkan splicing sama sekali, sementara spacer yang
membiarkan struktur itu melipat independen menaikkan efisiensi splicing dari
46% ke 87%. Itulah sebabnya aksesibilitas wilayah-wilayah ini diberi bobot
terbesar pada skor komposit.

In [ ]:
regions = {}
for name in ("E1", "E2", "IRES", "ORF", "intron_5half", "intron_3half"):
    if name in asm.precursor_coords:
        regions[name] = asm.precursor_coords[name]
e1a, e1b = asm.precursor_coords["E1"]
e2a, _ = asm.precursor_coords["E2"]
regions["splice5_site"] = (e1b - 9, e1b + 10)
regions["splice3_site"] = (e2a - 10, e2a + 9)
for key in ("intron_5half:IGS", "intron_3half:IGS"):
    if key in asm.precursor_coords:
        regions[key.replace(":", "_")] = asm.precursor_coords[key]

for t in (37.0, 55.0):
    print(f"--- {t:g} C")
    acc = access.accessibility(asm.precursor, regions, ulength=10, window=80, temperature_c=t)
    for name, v in sorted(acc.items(), key=lambda kv: kv[1]["mean"] or 0):
        if v["mean"] is None:
            continue
        print(f"  {name:20s} rerata {v['mean']:.5f}  min {v['min']:.5f}")

## 7. Empat pemeriksaan keras

1. **Domain IRES tidak rusak** oleh lipatan konstruk — dibandingkan status
   lipatan IRES di dalam konstruk vs saat IRES dilipat sendiri.
2. **Tidak ada dsRNA panjang** (> 30 bp beruntun) — proksi aktivasi PKR/RIG-I.
3. **Start codon tidak tertutup** struktur.
4. **Tidak ada motif pengganggu**: terminator T7 prematur, situs Tipe IIS
   internal, homopolimer panjang, jendela GC ekstrem.

Catatan kejujuran metode: metrik IRES di sini adalah **proksi struktural**.
Model energi ViennaRNA tidak memodelkan pseudoknot, sedangkan IRES pikornaviral
dan struktur tersier intron grup I bergantung padanya. Jangan laporkan metrik
ini sebagai prediksi fungsi translasi.

In [ ]:
ia, ib = asm.circle_coords["IRES"]
ires_seq = asm.circle[ia - 1:ib]
for t in (37.0, 55.0):
    alone = fold.fold(ires_seq, circular=False, temperature_c=t)
    whole = fold.fold(asm.circle, circular=True, temperature_c=t)
    ident = fold.subsequence_structure_identity(whole.mfe_structure, alone.mfe_structure, ia - 1)
    ds = access.longest_dsrna(whole.mfe_structure)
    print(f"@{t:g} C  identitas lipatan IRES {ident:.3f}   "
          f"heliks dsRNA terpanjang {ds['longest_bp']} bp   "
          f"pelanggaran >=30 bp: {len(access.dsrna_violations(whole.mfe_structure, 30))}")

print()
print("Pemindaian motif pada PREKURSOR (relevan untuk IVT dan kloning):")
print(json.dumps({k: v for k, v in motifs.summarize(asm.precursor).items()
                  if k in ("restriction", "m6a_sites", "m6a_density_per_100nt", "avoid_motifs")},
                 indent=2))
print("Terminator T7 prematur:", len(motifs.scan_t7_terminator(asm.precursor)))
print("Jendela GC di luar 30-70%:", len(motifs.gc_window_violations(asm.precursor)))

## 8. Skor komposit dan keputusan GO / NO-GO

Struktur keputusan dua lapis:

**Gerbang keras** — satu pelanggaran memaksa NO-GO, karena kegagalannya tidak
bisa dikompensasi kriteria lain (contoh: situs BsaI internal membuat perakitan
Golden Gate gagal, seberapa bagus pun lipatannya).

**Tujuh kriteria berbobot** dengan jumlah bobot 1,00. Semua bobot dan ambang
ditandai **[PERKIRAAN]**: ditetapkan pipeline ini, belum dikalibrasi terhadap
efisiensi sirkularisasi hasil laboratorium. Karena itu sel berikutnya
menjalankan **uji sensitivitas**.

In [ ]:
m = score.measure(asm, temperatures=(37.0, 55.0))
dec = score.score(m)

print(f"KEPUTUSAN: {dec.verdict}    skor komposit {dec.composite:.4f}")
print(f"Ambang: GO >= {dec.thresholds['go']}, GO-BERSYARAT >= {dec.thresholds['conditional']}")
print()
print("Gerbang keras:")
for k, v in dec.hard_gates.items():
    print(f"  [{'LULUS' if v['passed'] else 'GAGAL'}] {k:34s} nilai={v['value']}  batas={v['limit']}")
print()
print(f"{'kriteria':30s} {'bobot':>6s} {'subskor':>8s} {'kontribusi':>11s}")
for c in dec.criteria:
    print(f"{c.name:30s} {c.weight:6.2f} {c.subscore:8.3f} {c.weight * c.subscore:11.3f}")
print()
for c in dec.criteria:
    print(f"- {c.name}: {c.rationale}")

### Uji sensitivitas terhadap bobot

200 set bobot dibangkitkan dengan jitter ±50% di sekitar bobot bawaan, lalu
keputusan dihitung ulang. Yang perlu dilihat: apakah **distribusi keputusan**
seragam. Bila terbelah, keputusan Anda bergantung pada pilihan bobot dan itu
harus dinyatakan eksplisit di dokumen TA.

In [ ]:
sens = score.sensitivity(m, n=200, seed=7)
print(json.dumps(sens, indent=2))
print()
if len(sens["verdict_counts"]) == 1:
    print("Keputusan KOKOH terhadap pilihan bobot.")
else:
    print("PERHATIAN: keputusan BERUBAH tergantung bobot. Laporkan sebagai temuan,")
    print("dan jangan menyajikan satu angka skor sebagai kesimpulan tunggal.")

## 9. Bila NO-GO atau GO-BERSYARAT: siklus optimasi

Optimasi multi-objektif Optuna (NSGA-II, 6 objektif) atas panjang dan komposisi
spacer, panjang lengan homologi, serta seed/bobot optimasi kodon.

Nilai yang dicari bukan satu titik optimum melainkan **front Pareto**; front itu
lalu diperingkat ulang dengan skor komposit Tahap 0.

Panjang lengan homologi yang diuji mencakup 9 dan 19 nt, yaitu nilai yang diuji
**[LITERATUR]** Wesselhoeft dkk. (2018): tanpa lengan homologi efisiensi
splicing 0%, dengan 9 nt menjadi 16%, dengan 19 nt menjadi 48%.

Sel ini butuh beberapa menit. Naikkan `n_trials` bila ada waktu.

In [ ]:
from circhbsag import optimize as opt

base = {k: v for k, v in parts.items() if k != "ORF"}
study, records = opt.run_study(base, protein, n_trials=12, seed=20260101,
                               codon_iterations=2000)
ranked = opt.rank_records(records)

print(f"{len(records)} trial, {len(study.best_trials)} di front Pareto.\n")
print(f"{'trial':>5s} {'skor':>7s} {'keputusan':<14s} {'spacer5':>8s} {'spacer3':>8s} "
      f"{'pustaka':<13s} {'HA':>4s}")
for r in ranked[:10]:
    p = r["params"]
    print(f"{r['trial']:5d} {r['composite_score']:7.4f} {r['verdict']:<14s} "
          f"{p['spacer5_len']:8d} {p['spacer3_len']:8d} {p['spacer_library']:<13s} {p['ha_len']:4d}")

terbaik = ranked[0]
print(f"\nTerbaik: trial {terbaik['trial']}, skor {terbaik['composite_score']:.4f} "
      f"({terbaik['verdict']})")
print("Bandingkan dengan skor desain awal:", f"{dec.composite:.4f}", f"({dec.verdict})")

## 10. Primer, probe RNase H, dan primer IVT

- **Primer divergen** dirancang pada titik **antipodal** BSJ. Pada templat
  linear kedua primer saling menjauh sehingga tidak ada produk; hanya templat
  sirkular memberi amplikon. Inilah dasar RT-PCR divergen sebagai bukti
  sirkularisasi.
- **Primer menumpang BSJ** hanya dapat menempel bila E1 dan E2 benar-benar
  tersambung.
- **Probe RNase H** dipilih dari wilayah yang aksesibilitasnya tinggi, berjarak
  ≥ 40 nt dari BSJ. Lingkaran utuh + satu probe menghasilkan **satu** pita
  linear; prekursor atau concatemer memberi pola berbeda.
- **Primer reverse IVT** dengan 2'-O-metil pada dua nukleotida 5' menekan
  penambahan nukleotida non-templat oleh T7 di ujung 3' transkrip.

Ukuran pita yang diharapkan dicetak agar dapat langsung dipakai menyiapkan gel.

In [ ]:
oligo = primers.design_all(asm)

print("PRIMER DIVERGEN (3 teratas)")
for p in oligo["divergent"]["pairs"][:3]:
    print(f"  F {p['left_seq']}  Tm {p['left_tm']}  |  R {p['right_seq']}  Tm {p['right_tm']}")
    print(f"    produk pada lingkaran ~{p['expected_product_on_circle_nt']} nt; "
          f"pada linear: {p['expected_product_on_linear']}")

print("\nPRIMER MENUMPANG BSJ")
for p in oligo["bsj"]["junction_spanning_primers"]:
    print(f"  {p['name']:22s} {p['seq']}  Tm {p['tm_primer3']}  hairpin {p['hairpin_tm']}")

print("\nPROBE RNase H")
for p in oligo["rnase_h"]["probes"]:
    print(f"  posisi {p['start_1based']:4d}-{p['end_1based']:4d}  {p['probe_dna_5to3']}  "
          f"Tm {p['tm']}  aksesibilitas {p['accessibility_plfold']:.4g}")
print(" ", oligo["rnase_h"]["interpretation"])

print("\nPRIMER REVERSE IVT")
print(" ", oligo["ivt_reverse"]["with_modifications"])
print(" ", oligo["ivt_reverse"]["modification"])

## 11. Filter statistik BSJ ala Szabo

Tiga kriteria yang dinilai per bacaan:

1. **Panjang jangkar** di kedua sisi sambungan — jangkar pendek mudah muncul
   secara kebetulan.
2. **Jumlah mismatch** — di bawah H1 mismatch mengikuti laju galat sekuensing;
   di bawah H0 lebih tinggi.
3. **Keunikan** — bacaan yang **seluruhnya dapat dijelaskan referensi linear**
   bukan bukti sirkularisasi, berapa pun panjang jangkarnya, karena satu
   pengulangan sekuens sudah cukup menghasilkan pola seperti sambungan balik.
   Bacaan semacam itu **dikeluarkan**, bukan sekadar diberi penalti.

Keluarannya posterior Bayes per bacaan dan posterior gabungan. Angka presisi di
bawah mengukur **filter statistiknya** pada bacaan simulasi, bukan kinerja
eksperimen.

Perhatikan `n_excluded_linear_explained`: bila besar, konstruk Anda memuat
pengulangan sekuens yang menurunkan daya deteksi BSJ dan sebaiknya didesain ulang.

In [ ]:
sim = bsj_stats.run_simulation(asm.circle, asm.precursor, n_circular=1500,
                                n_linear=1500, read_len=150, anchor=20, seed=20260101)
print(json.dumps(sim["parameters"], indent=2))
print("\nBacaan yang diharapkan melintasi BSJ:", sim["expected_bsj_crossing_reads"])
print("Dikeluarkan karena dapat dijelaskan referensi linear:",
      sim["calls"]["n_excluded_linear_explained"])
print()
print(json.dumps({k: v for k, v in sim["support"].items() if k != "per_read"},
                 indent=2, ensure_ascii=False))
print("\nAlat eksternal untuk validasi silang:")
for nama, jalur in sim["external_tools_available"].items():
    print(f"  {nama:14s} {jalur or 'tidak terpasang'}")
print()
print("Perintah STAR chimeric yang setara bila STAR terpasang:")
print(" ", bsj_stats.write_star_cmd("reads.fq", "star_index/", "out/"))

## 12. Homologi off-target

Pemeriksaan ini **tidak mengarang hasil**: bila `blastn` dan basis datanya tidak
tersedia, statusnya `SKIPPED` dan perintah yang harus Anda jalankan dicetak.
Alternatif tanpa jaringan: filter k-mer 21 nt terhadap FASTA lokal (mis. CDS
manusia). k = 21 dipilih karena 4²¹ jauh melebihi ukuran transkriptom manusia,
sehingga satu k-mer 21 nt yang sama sudah menandakan homologi lokal nyata.

In [ ]:
from circhbsag import offtarget

print(json.dumps(offtarget.blast_available(), indent=2))
hasil_ot = offtarget.check_construct(asm, db=None, reference_fasta=None,
                                     out_dir=ROOT / "results" / "notebook")
for nama, v in hasil_ot.items():
    print(f"\n{nama}: {v['status']}")
    if v["status"] == "SKIPPED":
        print("  ", v["reason"])

## 13. Tulis laporan

Tiga format: JSON (data lengkap, untuk dianalisis ulang), Markdown (untuk
disalin ke dokumen TA), dan HTML mandiri (untuk dibagikan ke pembimbing).

Setiap berkas memuat blok provenans di atas, versi tiap alat, dan seed.

In [ ]:
payload = {
    "measurements": m,
    "decision": dec.to_dict(),
    "sensitivity": sens,
    "codon": hasil_kodon,
    "primers": oligo,
    "bsj_simulation": sim,
    "offtarget": hasil_ot,
    "permuted_template": tpl,
    "catatan_provenans_operator": CATATAN_PROVENANS,
}
rep = report.envelope(payload, provenance=meta, seed=20260101)
paths = report.write_all(rep, ROOT / "results" / "notebook", stem="tahap0_notebook")

print("LABEL DATA:", rep["data_label"])
print()
for k, v in paths.items():
    print(f"{k:9s} {v}")

## 14. Apa yang BOLEH dan TIDAK BOLEH disimpulkan

**Boleh:**

- Konstruk ini lolos atau tidak lolos gerbang keras tertentu — ini pemeriksaan
  sekuens yang deterministik dan dapat diperiksa ulang siapa pun.
- Peringkat relatif antar-varian desain pada kriteria termodinamika yang sama.
- Prediksi ukuran pita untuk RT-PCR divergen dan uji RNase H.

**Tidak boleh:**

- Menyatakan konstruk "akan tersirkularisasi dengan efisiensi X%". Pipeline ini
  tidak dikalibrasi terhadap efisiensi laboratorium; korelasi skor dengan hasil
  lab hanya boleh dilaporkan setelah data pilot ada, dan sebagai eksploratif.
- Menyatakan IRES "akan berfungsi". Metrik IRES di sini proksi struktural, dan
  model energi tanpa pseudoknot tidak dapat menilai fungsi IRES.
- Memakai nilai CAI absolut sebagai angka final sebelum tabel penggunaan kodon
  diganti dengan unduhan terverifikasi (`circhbsag codon-table`).

**Langkah berikutnya bila GO:** sintesis fragmen, perakitan, lalu bukti
sirkularisasi ortogonal (RNase R terstandar dengan titrasi dan spike-in, RT-PCR
divergen + sekuensing Sanger sambungan, pergeseran migrasi, uji RNase H).
Keputusan GO dari notebook ini **tidak** menggantikan satu pun dari uji itu.